In [1]:
import polars as pl

In [40]:
path = '../data/raw/ai4i2020.csv'
df_raw = pl.read_csv(path)
print('Dataset loaded')
print(f'Rows: {df_raw.shape[0]}')
print(f'Columns: {df_raw.shape[1]}')

df_raw.sample(10)

Dataset loaded
Rows: 10000
Columns: 14


UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
i64,str,str,f64,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64
3184,"""L50363""","""L""",300.1,309.3,1616,32.0,199,0,0,0,0,0,0
2364,"""L49543""","""L""",299.3,308.5,1510,45.4,72,0,0,0,0,0,0
9697,"""L56876""","""L""",298.9,309.9,1404,39.8,62,0,0,0,0,0,0
975,"""L48154""","""L""",296.0,306.6,1511,35.2,119,0,0,0,0,0,0
9627,"""M24486""","""M""",299.0,310.0,1341,58.9,126,0,0,0,0,0,0
5305,"""L52484""","""L""",303.9,313.1,1425,49.3,199,0,0,0,0,0,0
6226,"""M21085""","""M""",301.2,311.0,1424,41.4,131,0,0,0,0,0,0
3645,"""L50824""","""L""",302.2,311.5,1548,32.4,81,0,0,0,0,0,0
7934,"""L55113""","""L""",300.8,311.8,1928,21.8,209,0,0,0,0,0,0


In [3]:
print('Columns:')
print(df_raw.columns)
print('\nData Quality')
df_raw.describe()

Columns:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Data Quality


statistic,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",10000.0,"""10000""","""10000""",10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5000.5,null,null,300.00493,310.00556,1538.7761,39.98691,107.951,0.0339,0.0046,0.0115,0.0095,0.0098,0.0019
"""std""",2886.89568,null,null,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
"""min""",1.0,"""H29424""","""H""",295.3,305.7,1168.0,3.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",2501.0,null,null,298.3,308.8,1423.0,33.2,53.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",5001.0,null,null,300.1,310.1,1503.0,40.1,108.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",7500.0,null,null,301.5,311.1,1612.0,46.8,162.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",10000.0,"""M24859""","""M""",304.5,313.8,2886.0,76.6,253.0,1.0,1.0,1.0,1.0,1.0,1.0


In [4]:
print('Separating all column by type:\n')
identifier = [
    'UDI', 'Product ID', 'Type'
]
failure = [
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF'
]
target = [
    'Machine failure'
]
operational = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',	'Torque [Nm]', 'Tool wear [min]'
]
print(f'Identifier: {identifier}')
print(f'Failure: {failure}')
print(f'Operational: {operational}')
print(f'Target: {target}')

Separating all column by type:

Identifier: ['UDI', 'Product ID', 'Type']
Failure: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
Operational: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target: ['Machine failure']


In [5]:
total_failures = df_raw.group_by('Machine failure').agg(
    pl.len().alias('total_records')
    ).with_columns(
        (pl.col('total_records') / pl.col('total_records').sum()).alias('percentage')
    )
print('Total failures:')
total_failures

Total failures:


Machine failure,total_records,percentage
i64,u32,f64
0,9661,0.9661
1,339,0.0339


In [6]:
import plotly.express as px

fig = px.bar(
    data_frame= total_failures, x= 'Machine failure', y= 'total_records'
)

fig.show()

In [7]:
count = df_raw.select(pl.col(failure).sum()).row(0)
total = df_raw.filter(pl.col('Machine failure') == 1).group_by('Machine failure').agg(pl.len()).row(0)[1]

failure_distribution = pl.DataFrame({
    'failure' : failure,
    'count': count,
}).with_columns(
    (pl.col('count') / total * 100).alias('percentage_failures'),
        (pl.col('count') / df_raw.height *100).alias('percentage_total')
).sort('count', descending= False)
failure_distribution


failure,count,percentage_failures,percentage_total
str,i64,f64,f64
"""RNF""",19,5.60472,0.19
"""TWF""",46,13.569322,0.46
"""PWF""",95,28.023599,0.95
"""OSF""",98,28.908555,0.98
"""HDF""",115,33.923304,1.15


In [8]:

fig = px.bar(failure_distribution, x= 'failure', y= 'count', title= 'Count of failure')
fig.show()

In [29]:
sum_machine_failure_errors = df_raw.filter(
    (pl.col(target) == 0) & (pl.sum_horizontal(failure) == 1)
).sum().select(
    pl.col(target),
    pl.col(failure)
).with_columns(
    pl.sum_horizontal(failure).alias('sum_failure')
)

sum_failure_errors = df_raw.filter(
    (pl.col(target) == 1) & (pl.sum_horizontal(failure) == 0)
).sum().select(
    pl.col(target),
    pl.col(failure)
).with_columns(
    pl.sum_horizontal(failure).alias('sum_failure')
)

print('Sum Machine Failures errors:')
print(sum_machine_failure_errors)
print('Sum Failure errors:')
print(sum_failure_errors)

for col in failure:
    anomalies = df_raw.filter((pl.col(col) == 1) & (pl.col(target) == 0))
    percentage = anomalies.shape[0] / df_raw.filter(pl.col(col) > 0 ).shape[0] *100
    if anomalies.shape[0] > 0:
        print(f'For {col}, total anomalies = {anomalies.shape[0]} with a percentage of {percentage}')

anomalies_mf = df_raw.filter((pl.col(target) == 1) & (pl.all_horizontal(pl.col(failure) == 0)))
percentage_mf = anomalies_mf.shape[0] / (df_raw.filter(pl.col(target) == 1)).shape[0] *100

print(f'For Machine Failure, total anomalies = {anomalies_mf.shape[0]} with a percentage of {percentage_mf}')

Sum Machine Failures errors:
shape: (1, 7)
┌─────────────────┬─────┬─────┬─────┬─────┬─────┬─────────────┐
│ Machine failure ┆ TWF ┆ HDF ┆ PWF ┆ OSF ┆ RNF ┆ sum_failure │
│ ---             ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---         │
│ i64             ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64         │
╞═════════════════╪═════╪═════╪═════╪═════╪═════╪═════════════╡
│ 0               ┆ 0   ┆ 0   ┆ 0   ┆ 0   ┆ 18  ┆ 18          │
└─────────────────┴─────┴─────┴─────┴─────┴─────┴─────────────┘
Sum Failure errors:
shape: (1, 7)
┌─────────────────┬─────┬─────┬─────┬─────┬─────┬─────────────┐
│ Machine failure ┆ TWF ┆ HDF ┆ PWF ┆ OSF ┆ RNF ┆ sum_failure │
│ ---             ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---         │
│ i64             ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64         │
╞═════════════════╪═════╪═════╪═════╪═════╪═════╪═════════════╡
│ 9               ┆ 0   ┆ 0   ┆ 0   ┆ 0   ┆ 0   ┆ 0           │
└─────────────────┴─────┴─────┴─────┴─────┴─────┴─────────────┘
For RNF, total anomalies = 

- Since anomalies in RNF represent 94% of all failures in this column, it's worth excluding it
- Having 9 rows with "Machine failure" but no corresponding failure type tells us nothing, so it's worth excluding them

In [38]:
df = df_raw.filter(
    ((pl.col(target) == 0) & pl.all_horizontal(pl.col(failure) == 0))
    | ((pl.col(target) == 1) & pl.any_horizontal(pl.col(failure) == 1))
)
total_lines = df_raw.shape[0]
new_total_lines = df.shape[0]
sum_failures = df_raw.filter(
    ((pl.col(target) == 1) & pl.all_horizontal(pl.col(failure) == 0))
    | ((pl.col(target) == 0) & pl.any_horizontal(pl.col(failure) == 1))
).shape[0]

print(f'Rows removed: {total_lines - new_total_lines} (expected: {sum_failures})')

Rows removed: 27 (expected: 27)


In [ ]:
try:
    df.write_csv('../data/processed/ai4i2020_new.csv')
    print('Dataset saved at /data/processed')
except:
    print('Fail to save')

Dataset saved at /data/processed
